## Historical Aerial Photograph Index Preparation

This notebook cleans and standardizes the original Minnesota Historical Aerial Photographs Online (MHAPO) point feature class and produces two datasets:
- A cleaned feature class containing the original useful metadata, corrected attributes, and unique photograph identifiers.
- A blank georeferencing index containing the subset of attributes required for the batch georeferencing workflow.

Both datasets share a common unique identifier (source_id) to support joins and subsequent updates.

The original source data are preserved without modification. Processing results and georeferencing status are managed separately from these datasets.

### Planning
- Preserve the original row number for provenance immediately after loading.
- Perform all cleaning and deduplication.
- Sort the final cleaned table.
- Assign permanent source_id values.
- Generate and validate cty_year_id.
- Derive both output tables from that same cleaned GeoDataFrame.

#### Sequence
- 01 Setup and configuration
- 02 Load and inspect source data
- 03 Remove empty fields and standardize field names
- 04 Normalize attribute values
- 05 Spatial county assignment and validation
- 06 Correct known URL errors
- 07 Investigate and resolve duplicate records
- 08 Sort records and generate permanent IDs
- 09 Calculate derived attributes
- 10 Construct the blank georeferencing index
- 11 Validate both output datasets
- 12 Export cleaned feature class and blank index

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

# Find the repository root by locating the src directory.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\eu01251115\LocalProjects\batch_georef


In [3]:
# Configure file paths
# This is separate from config.yml, to isolate this from batch processing and facilitate manual re-runs

DATA_DIR = PROJECT_ROOT / "data" / "historic_airphoto_index"

SOURCE_PATH = (DATA_DIR / "shapefiles" / "umn_airphoto_index_original.shp")

COUNTIES_PATH = (DATA_DIR / "shapefiles" / "Minnesota_Counties.shp")

CLEANED_PATH = DATA_DIR / "umn_index_cleaned.gpkg"

INDEX_PATH = DATA_DIR / "georef_index_blank.gpkg"

LAYER = "photos"

for path in [SOURCE_PATH, COUNTIES_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

In [4]:
# Load and inspect source data

gdf_original = gpd.read_file(SOURCE_PATH)
counties = gpd.read_file(COUNTIES_PATH)

print("Photo records:", len(gdf_original))
print("Photo index CRS:", gdf_original.crs)

print("County records:", len(counties))
print("County CRS:", counties.crs)

Photo records: 115749
Photo index CRS: EPSG:3857
County records: 87
County CRS: EPSG:3857
